In [2]:
# WHAT THIS DOES: imports, settings, output folder.
# Run once at the start of your session.

import requests
import json
import time
import re
import csv
from pathlib import Path

TARGET_PROVIDERS = ["Places Leisure", "Everyone Active", "Serco Leisure"]
CATALOG_COLLECTION_URL = "https://openactive.io/data-catalogs/data-catalog-collection.jsonld"

HEADERS = {
    "User-Agent": "LondonSport-MastersProject/1.0 (contact: your-email@example.com)",
    "Accept": "application/json",
}

OUTPUT_DIR = Path("./openactive_data/person2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_PAGES = 0.3
MAX_PAGES_PER_FEED = None  # set e.g. 10 while testing, to stop early

In [3]:
# WHAT THIS DOES: defines functions to find dataset site URLs in the
# OpenActive catalog, and to pull JSON-LD metadata out of a dataset site's HTML.
# Nothing runs yet — just defining tools for Cell 3.

def get_dataset_site_urls():
    resp = requests.get(CATALOG_COLLECTION_URL, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    collection = resp.json()

    dataset_urls = []
    for catalog_ref in collection.get("hasPart", []):
        catalog_url = catalog_ref if isinstance(catalog_ref, str) else catalog_ref.get("@id")
        if not catalog_url:
            continue
        try:
            cat_resp = requests.get(catalog_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            cat_resp.raise_for_status()
            catalog = cat_resp.json()
        except Exception as e:
            print(f"  [skip] couldn't fetch catalog {catalog_url}: {e}")
            continue
        for dataset in catalog.get("dataset", []):
            ds_url = dataset if isinstance(dataset, str) else dataset.get("url") or dataset.get("@id")
            if ds_url:
                dataset_urls.append(ds_url)
    return dataset_urls


def extract_jsonld_from_html(html, base_url):
    matches = re.findall(
        r'<script[^>]+type=["\']application/ld\+json["\'][^>]*>(.*?)</script>',
        html, re.DOTALL | re.IGNORECASE
    )
    for m in matches:
        try:
            data = json.loads(m.strip())
            if data.get("@type") == "Dataset":
                return data
        except json.JSONDecodeError:
            continue
    return None

In [4]:
# WHAT THIS DOES: actually fetches the catalog collection and checks every
# dataset site's name against your 3 target providers. This is the slowest
# step since it has to download ~150 pages. Run it once, then re-use the
# 'provider_jsonlds' result in later cells without re-running this.

def find_provider_dataset_sites(target_names):
    print("Discovering dataset site URLs...")
    dataset_urls = get_dataset_site_urls()
    print(f"  Found {len(dataset_urls)} dataset sites total.\n")

    found = {}
    targets_lower = [t.lower() for t in target_names]

    for i, url in enumerate(dataset_urls):
        if i % 25 == 0:
            print(f"  checked {i}/{len(dataset_urls)}...")
        try:
            r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
        except Exception:
            continue

        jsonld = extract_jsonld_from_html(r.text, url)
        if not jsonld:
            continue

        name = jsonld.get("name", "") or jsonld.get("publisher", {}).get("name", "")
        name_lower = name.lower()

        for target, target_lower in zip(target_names, targets_lower):
            if target_lower in name_lower and target not in found:
                print(f"  Matched: '{name}' -> '{target}' ({url})")
                found[target] = jsonld

    missing = set(target_names) - set(found.keys())
    if missing:
        print(f"\n  WARNING: not auto-matched: {missing}")
        print("  -> check exact names on https://www.findopendata.openactive.io/")

    return found


provider_jsonlds = find_provider_dataset_sites(TARGET_PROVIDERS)

Discovering dataset site URLs...
  Found 173 dataset sites total.

  checked 0/173...
  Matched: 'Everyone Active Sessions and Facilities' -> 'Everyone Active' (https://data.everyoneactive.com/OpenActive/)
  Matched: 'Places Leisure Sessions and Facilities' -> 'Places Leisure' (https://placesleisure.gs-signature.cloud/OpenActive/)
  checked 25/173...
  Matched: 'Serco Leisure Sessions and Facilities' -> 'Serco Leisure' (https://serco-openactive.legendonlineservices.co.uk/OpenActive)
  checked 50/173...
  checked 75/173...
  checked 100/173...
  checked 125/173...
  checked 150/173...


In [5]:
# WHAT THIS DOES: a fallback for any provider Cell 3 couldn't auto-match.
# Look the provider up on findopendata.openactive.io, view page source,
# find the <script type="application/ld+json"> block, copy the feed URLs
# from its "distribution" array. Leave empty if Cell 3 found everything.

MANUAL_FEED_URLS = {
    # "Serco Leisure": {"SessionSeries": "https://example.com/feeds/session-series"},
}

In [6]:
# WHAT THIS DOES: pulls the actual feed URLs (one per data type, e.g.
# SessionSeries, FacilityUse) out of each provider's JSON-LD. Fast — no
# network calls, just reads what's already in provider_jsonlds.

def get_feed_urls(jsonld):
    feeds = {}
    for dist in jsonld.get("distribution", []):
        feed_type = dist.get("additionalType", "Unknown")
        feed_type_short = feed_type.rstrip("/").split("/")[-1]
        feed_url = dist.get("contentUrl")
        if feed_url:
            feeds[feed_type_short] = feed_url
    return feeds


provider_feeds = {}
for provider_name in TARGET_PROVIDERS:
    if provider_name in provider_jsonlds:
        provider_feeds[provider_name] = get_feed_urls(provider_jsonlds[provider_name])
    elif provider_name in MANUAL_FEED_URLS:
        provider_feeds[provider_name] = MANUAL_FEED_URLS[provider_name]
    else:
        provider_feeds[provider_name] = {}
        print(f"No feeds for {provider_name} — fill in MANUAL_FEED_URLS in Cell 4.")

print(provider_feeds)  # sanity check before harvesting anything

{'Places Leisure': {'CourseInstance': 'https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-course-instance', 'SessionSeries': 'https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-session-series', 'ScheduledSession': 'https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-scheduled-sessions', 'FacilityUse': 'https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-facility-uses', 'Slot': 'https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-slots'}, 'Everyone Active': {'CourseInstance': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-course-instance', 'SessionSeries': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-session-series', 'ScheduledSession': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-scheduled-sessions', 'FacilityUse': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-facility-uses', 'Slot': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-l

In [8]:
# WHAT THIS DOES: defines the actual harvesting function — follows the
# RPDE 'next' link page by page until it catches up to the live edge of
# the feed. This replaces the broken `openactive` package. Doesn't run yet.

def harvest_rpde_feed(feed_url, provider_name, feed_type):
    all_items = []
    current_url = feed_url
    page = 0
    seen_urls = set()

    print(f"\n  Harvesting {provider_name} / {feed_type}")
    print(f"  Start URL: {feed_url}")

    while True:
        page += 1
        if MAX_PAGES_PER_FEED and page > MAX_PAGES_PER_FEED:
            print(f"    Hit MAX_PAGES_PER_FEED cap ({MAX_PAGES_PER_FEED}), stopping early.")
            break

        try:
            resp = requests.get(current_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"    [ERROR] page {page}: {e} -- stopping.")
            break

        try:
            data = resp.json()
        except json.JSONDecodeError:
            print(f"    [ERROR] page {page}: not valid JSON -- stopping.")
            break

        items = data.get("items", [])
        next_url = data.get("next")
        kept = [it for it in items if it.get("state") != "deleted" and it.get("data")]
        all_items.extend(kept)

        if page % 20 == 0 or not items:
            print(f"    page {page}: +{len(kept)} (total: {len(all_items)})")

        if not items and next_url == current_url:
            print(f"    Reached live edge after {page} pages. Total: {len(all_items)}")
            break
        if not next_url or next_url in seen_urls:
            print(f"    No further 'next' / loop detected. Total: {len(all_items)}")
            break

        seen_urls.add(current_url)
        current_url = next_url
        time.sleep(SLEEP_BETWEEN_PAGES)

    return all_items

In [9]:
# WHAT THIS DOES: defines two ways to save harvested items — raw JSON
# (full fidelity) and a flattened CSV (quick inspection / coverage mapping).

def save_items(items, provider_name, feed_type):
    safe = lambda s: re.sub(r"[^a-zA-Z0-9_-]", "_", s)
    out_path = OUTPUT_DIR / f"{safe(provider_name)}__{safe(feed_type)}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)
    print(f"    Saved {len(items)} items -> {out_path}")
    return out_path


def flatten_for_csv(items, provider_name, feed_type):
    safe = lambda s: re.sub(r"[^a-zA-Z0-9_-]", "_", s)
    out_path = OUTPUT_DIR / f"{safe(provider_name)}__{safe(feed_type)}.csv"
    rows = []
    for item in items:
        d = item.get("data", {})
        location = d.get("location", {}) or {}
        geo = location.get("geo", {}) or {}
        address = location.get("address", {}) or {}
        rows.append({
            "provider": provider_name, "feed_type": feed_type,
            "item_id": item.get("id"), "name": d.get("name"),
            "lat": geo.get("latitude"), "lon": geo.get("longitude"),
            "address_locality": address.get("addressLocality"),
            "postal_code": address.get("postalCode"),
            "modified": item.get("modified"),
        })
    if rows:
        with open(out_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f"    Saved CSV -> {out_path}")
    return out_path

In [10]:
# WHAT THIS DOES: actually harvests one provider's feeds. This is the
# longest-running cell, since it pages through live data. Change
# PROVIDER_TO_RUN and re-run this cell for each of your 3 providers,
# rather than looping all of them in one go.

PROVIDER_TO_RUN = "Places Leisure"  # <-- change this each time

feeds = provider_feeds.get(PROVIDER_TO_RUN, {})
if not feeds:
    print(f"No feeds found for {PROVIDER_TO_RUN} — check Cells 3-5.")
else:
    for feed_type, feed_url in feeds.items():
        items = harvest_rpde_feed(feed_url, PROVIDER_TO_RUN, feed_type)
        if items:
            save_items(items, PROVIDER_TO_RUN, feed_type)
            flatten_for_csv(items, PROVIDER_TO_RUN, feed_type)


  Harvesting Places Leisure / CourseInstance
  Start URL: https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-course-instance
    page 2: +0 (total: 4)
    Reached live edge after 2 pages. Total: 4
    Saved 4 items -> openactive_data\person2\Places_Leisure__CourseInstance.json
    Saved CSV -> openactive_data\person2\Places_Leisure__CourseInstance.csv

  Harvesting Places Leisure / SessionSeries
  Start URL: https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-session-series
    page 20: +499 (total: 9981)
    page 40: +0 (total: 18994)
    Reached live edge after 40 pages. Total: 18994
    Saved 18994 items -> openactive_data\person2\Places_Leisure__SessionSeries.json
    Saved CSV -> openactive_data\person2\Places_Leisure__SessionSeries.csv

  Harvesting Places Leisure / ScheduledSession
  Start URL: https://opendata.leisurecloud.live/api/feeds/PlacesLeisure-live-scheduled-sessions
    page 20: +499 (total: 3157)
    page 40: +499 (total: 13091)
    page 60: 

In [9]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\Places_Leisure__Slot.csv")
print(len(df))  # should print 154238

154238


In [11]:
import json

with open(r"C:\Users\Hp\Downloads\Project 2026 DS\Places_Leisure__ScheduledSession.json") as f:
    sessions = json.load(f)

print(json.dumps(sessions[0], indent=2))

{
  "id": "00000000000200033070",
  "modified": 274740894,
  "kind": "ScheduledSession",
  "state": "updated",
  "data": {
    "@context": [
      "https://openactive.io/",
      "https://openactive.io/ns-beta"
    ],
    "@type": "ScheduledSession",
    "@id": "https://placesleisure.gs-signature.cloud/OpenActive/api/scheduled-sessions/200033070",
    "startDate": "2026-06-23T12:00:00+00:00",
    "identifier": 200033070,
    "endDate": "2026-06-23T13:00:00+00:00",
    "superEvent": "https://placesleisure.gs-signature.cloud/OpenActive/api/session-series/20721300SGG0126",
    "duration": "PT1H",
    "maximumAttendeeCapacity": 5,
    "remainingAttendeeCapacity": 5,
    "beta:sportsActivityLocation": [
      {
        "@type": "SportsActivityLocation",
        "name": "Gym Sessions"
      }
    ]
  }
}


In [12]:
import json

with open(r"C:\Users\Hp\Downloads\Project 2026 DS\Places_Leisure__SessionSeries.json") as f:
    series = json.load(f)

print(json.dumps(series[0], indent=2))


{
  "id": "20611030SST0225",
  "modified": 40227870,
  "kind": "SessionSeries",
  "state": "updated",
  "data": {
    "@context": [
      "https://openactive.io/",
      "https://openactive.io/ns-beta"
    ],
    "@type": "SessionSeries",
    "@id": "https://placesleisure.gs-signature.cloud/OpenActive/api/session-series/20611030SST0225",
    "eventSchedule": [
      {
        "@type": "PartialSchedule",
        "byDay": [
          "https://schema.org/Monday"
        ],
        "duration": "PT2H",
        "endTime": "12:30",
        "scheduleTimezone": "Europe/London",
        "startDate": "2025-02-10",
        "startTime": "10:30"
      }
    ],
    "identifier": "20611030SST0225",
    "name": "F A Badminton",
    "attendeeInstructions": "This is a great way to meet new people and play single or double Badminton. This session is for ages 50 and over.",
    "category": [
      "Activity Sessions"
    ],
    "duration": "PT2H",
    "location": {
      "@type": "Place",
      "identifier

In [13]:
print(json.dumps(series[0]["data"].get("location", "NOT FOUND"), indent=2))

{
  "@type": "Place",
  "identifier": "206",
  "name": "Eclipse Leisure Centre",
  "description": "We have a range of pool activities at Eclipse Leisure Centre. If you wish to learn how to swim or want to enjoy a lane swim session, we have something for you!\n\nOur gym accommodates to a variety of abilities and goals, helping our members stay active however they like. If group workout classes are more your thing, we offer classes such as BODYPUMP, Yoga and Group Cycling across all intensities.\u00a0\n\nWe also work with National Governing Bodies to ensure our centre has fun, engaging sports. Get stuck into trampolining, make a hit on one of our badminton courts or enjoy a match on one of our rooftop football pitches! \n\nFor your little ones, we have an exciting range of climbing walls where they can unleash their inner adventurer.\n\n",
  "address": {
    "@type": "PostalAddress",
    "addressCountry": "GB",
    "addressLocality": "Staines-Upon-Thames",
    "addressRegion": "Staines",

In [14]:
loc = series[0]["data"]["location"]
print("geo" in loc, list(loc.keys()))

True ['@type', 'identifier', 'name', 'description', 'address', 'amenityFeature', 'geo', 'image', 'openingHoursSpecification', 'telephone', 'url', 'beta:formattedDescription']


In [15]:
import json

with open(r"C:\Users\Hp\Downloads\Project 2026 DS\Places_Leisure__Slot.json") as f:
    slots = json.load(f)

print(json.dumps(slots[0], indent=2))

{
  "id": "095A001006_2026-06-22T08-00-00",
  "modified": 274540213,
  "kind": "FacilityUse/Slot",
  "state": "updated",
  "data": {
    "@context": [
      "https://openactive.io/",
      "https://openactive.io/ns-beta"
    ],
    "@type": "Slot",
    "@id": "https://placesleisure.gs-signature.cloud/OpenActive/api/slots/095A001006_2026-06-22T08-00-00",
    "identifier": "095A001006_2026-06-22T08-00-00",
    "duration": "PT1H",
    "facilityUse": "https://placesleisure.gs-signature.cloud/OpenActive/api/facility-uses/095A001006",
    "maximumUses": 1,
    "offers": [
      {
        "@type": "Offer",
        "price": 0,
        "priceCurrency": "GBP"
      }
    ],
    "remainingUses": 0,
    "startDate": "2026-06-22T07:00:00+00:00",
    "endDate": "2026-06-22T08:00:00+00:00",
    "beta:sportsActivityLocation": [
      {
        "@type": "SportsActivityLocation",
        "name": [
          "Court 1",
          "Court 2",
          "Court 3",
          "Court 4"
        ],
        "iden

In [59]:
#code for getting everyone active


import requests, json, time, re, csv
from pathlib import Path

# --- Config ---
TARGET_PROVIDERS = ["Everyone Active", "Serco Leisure"]  # skip Places Leisure, already done
CATALOG_COLLECTION_URL = "https://openactive.io/data-catalogs/data-catalog-collection.jsonld"
HEADERS = {"User-Agent": "LondonSport-MastersProject/1.0 (contact: your-email@example.com)", "Accept": "application/json"}
OUTPUT_DIR = Path(r"C:\Users\Hp\Downloads\Project 2026 DS")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_PAGES = 0.3

# --- Discovery functions ---
def get_dataset_site_urls():
    resp = requests.get(CATALOG_COLLECTION_URL, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    collection = resp.json()
    dataset_urls = []
    for catalog_ref in collection.get("hasPart", []):
        catalog_url = catalog_ref if isinstance(catalog_ref, str) else catalog_ref.get("@id")
        if not catalog_url:
            continue
        try:
            cat_resp = requests.get(catalog_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            cat_resp.raise_for_status()
            catalog = cat_resp.json()
        except Exception:
            continue
        for dataset in catalog.get("dataset", []):
            ds_url = dataset if isinstance(dataset, str) else dataset.get("url") or dataset.get("@id")
            if ds_url:
                dataset_urls.append(ds_url)
    return dataset_urls

def extract_jsonld_from_html(html):
    matches = re.findall(r'<script[^>]+type=["\']application/ld\+json["\'][^>]*>(.*?)</script>', html, re.DOTALL | re.IGNORECASE)
    for m in matches:
        try:
            data = json.loads(m.strip())
            if data.get("@type") == "Dataset":
                return data
        except json.JSONDecodeError:
            continue
    return None

def find_provider_dataset_sites(target_names):
    print("Discovering dataset site URLs...")
    dataset_urls = get_dataset_site_urls()
    print(f"  Found {len(dataset_urls)} dataset sites.\n")
    found = {}
    targets_lower = [t.lower() for t in target_names]
    for i, url in enumerate(dataset_urls):
        if i % 25 == 0:
            print(f"  checked {i}/{len(dataset_urls)}...")
        try:
            r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
        except Exception:
            continue
        jsonld = extract_jsonld_from_html(r.text)
        if not jsonld:
            continue
        name = (jsonld.get("name", "") or jsonld.get("publisher", {}).get("name", "")).lower()
        for target, target_lower in zip(target_names, targets_lower):
            if target_lower in name and target not in found:
                print(f"  Matched: '{jsonld.get('name')}' -> '{target}'")
                found[target] = jsonld
    return found

def get_feed_urls(jsonld):
    feeds = {}
    for dist in jsonld.get("distribution", []):
        feed_type = dist.get("additionalType", "Unknown").rstrip("/").split("/")[-1]
        feed_url = dist.get("contentUrl")
        if feed_url:
            feeds[feed_type] = feed_url
    return feeds

# --- Paginator ---
def harvest_rpde_feed(feed_url, provider_name, feed_type):
    all_items = []
    current_url = feed_url
    page = 0
    seen_urls = set()
    print(f"\n  Harvesting {provider_name} / {feed_type}\n  Start: {feed_url}")
    while True:
        page += 1
        try:
            resp = requests.get(current_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"    [ERROR] page {page}: {e} -- stopping.")
            break
        items = data.get("items", [])
        next_url = data.get("next")
        kept = [it for it in items if it.get("state") != "deleted" and it.get("data")]
        all_items.extend(kept)
        if page % 20 == 0 or not items:
            print(f"    page {page}: +{len(kept)} (total: {len(all_items)})")
        if not items and next_url == current_url:
            print(f"    Reached live edge. Total: {len(all_items)}")
            break
        if not next_url or next_url in seen_urls:
            break
        seen_urls.add(current_url)
        current_url = next_url
        time.sleep(SLEEP_BETWEEN_PAGES)
    return all_items



def save_items(items, provider_name, feed_type):
    safe = lambda s: re.sub(r"[^a-zA-Z0-9_-]", "_", s)
    provider_folder = OUTPUT_DIR / safe(provider_name)
    provider_folder.mkdir(parents=True, exist_ok=True)  # creates the subfolder if it doesn't exist
    out_path = provider_folder / f"{safe(feed_type)}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)
    print(f"    Saved {len(items)} items -> {out_path}")
    return out_path

# --- Run discovery + harvest for Everyone Active and Serco Leisure ---
provider_jsonlds = find_provider_dataset_sites(TARGET_PROVIDERS)
provider_feeds = {p: get_feed_urls(provider_jsonlds[p]) for p in TARGET_PROVIDERS if p in provider_jsonlds}
print("\nFeeds found:", provider_feeds)

Discovering dataset site URLs...
  Found 172 dataset sites.

  checked 0/172...
  Matched: 'Everyone Active Sessions and Facilities' -> 'Everyone Active'
  checked 25/172...
  Matched: 'Serco Leisure Sessions and Facilities' -> 'Serco Leisure'
  checked 50/172...
  checked 75/172...
  checked 100/172...
  checked 125/172...
  checked 150/172...

Feeds found: {'Everyone Active': {'CourseInstance': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-course-instance', 'SessionSeries': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-session-series', 'ScheduledSession': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-scheduled-sessions', 'FacilityUse': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-facility-uses', 'Slot': 'https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-slots'}, 'Serco Leisure': {'SessionSeries': 'https://serco-openactive.legendonlineservices.co.uk/api/sessions', 'FacilityUse': 'https://se

In [ ]:
PROVIDER_TO_RUN = "Everyone Active"
for feed_type, feed_url in provider_feeds.get(PROVIDER_TO_RUN, {}).items():
    items = harvest_rpde_feed(feed_url, PROVIDER_TO_RUN, feed_type)
    if items:
        save_items(items, PROVIDER_TO_RUN, feed_type)




  Harvesting Everyone Active / CourseInstance
  Start: https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-course-instance
    page 20: +499 (total: 9981)
    page 40: +499 (total: 19961)
    page 55: +0 (total: 26514)
    Reached live edge. Total: 26514
    Saved 26514 items -> C:\Users\Hp\Downloads\Project 2026 DS\Everyone_Active\CourseInstance.json

  Harvesting Everyone Active / SessionSeries
  Start: https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-session-series
    page 20: +499 (total: 9981)
    page 40: +499 (total: 19961)
    page 60: +499 (total: 29941)
    page 80: +499 (total: 39921)
    page 100: +499 (total: 49901)
    page 120: +499 (total: 59881)
    page 139: +0 (total: 68551)
    Reached live edge. Total: 68551
    Saved 68551 items -> C:\Users\Hp\Downloads\Project 2026 DS\Everyone_Active\SessionSeries.json

  Harvesting Everyone Active / ScheduledSession
  Start: https://opendata.leisurecloud.live/api/feeds/EveryoneActive-live-scheduled

In [57]:
import time

def harvest_rpde_feed(feed_url, provider_name, feed_type):
    session = requests.Session()
    session.headers.update(HEADERS)

    all_items = []
    current_url = feed_url
    page = 0
    seen_urls = set()
    print(f"\n  Harvesting {provider_name} / {feed_type}\n  Start: {feed_url}")

    while True:
        page += 1
        try:
            resp = session.get(current_url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 403:
                print(f"    [403] page {page} -- retrying once after a pause...")
                time.sleep(3)
                resp = session.get(current_url, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"    [ERROR] page {page}: {e} -- stopping.")
            break

        items = data.get("items", [])
        next_url = data.get("next")
        kept = [it for it in items if it.get("state") != "deleted" and it.get("data")]
        all_items.extend(kept)

        if page % 20 == 0 or not items:
            print(f"    page {page}: +{len(kept)} (total: {len(all_items)})")
        if not items and next_url == current_url:
            print(f"    Reached live edge. Total: {len(all_items)}")
            break
        if not next_url or next_url in seen_urls:
            break
        seen_urls.add(current_url)
        current_url = next_url
        time.sleep(5.0)  # increased pause — Serco's server seems stricter

    return all_items

In [60]:
PROVIDER_TO_RUN = "Serco Leisure"
for feed_type, feed_url in provider_feeds.get(PROVIDER_TO_RUN, {}).items():
    items = harvest_rpde_feed(feed_url, PROVIDER_TO_RUN, feed_type)
    if items:
        save_items(items, PROVIDER_TO_RUN, feed_type)


  Harvesting Serco Leisure / SessionSeries
  Start: https://serco-openactive.legendonlineservices.co.uk/api/sessions
    [ERROR] page 5: 403 Client Error: Forbidden for url: https://serco-openactive.legendonlineservices.co.uk/api/sessions?afterTimestamp=10586883&afterId=12163340 -- stopping.

  Harvesting Serco Leisure / FacilityUse
  Start: https://serco-openactive.legendonlineservices.co.uk/api/facility-uses
    page 3: +0 (total: 132)
    Reached live edge. Total: 132
    Saved 132 items -> C:\Users\Hp\Downloads\Project 2026 DS\Serco_Leisure\FacilityUse.json

  Harvesting Serco Leisure / Slot
  Start: https://serco-openactive.legendonlineservices.co.uk/api/facility-uses/events
    [ERROR] page 4: 403 Client Error: Forbidden for url: https://serco-openactive.legendonlineservices.co.uk/api/facility-uses/events?afterTimestamp=10586837&afterId=12394579 -- stopping.


In [ ]:
#secro leisure(took too long to run so interupted)
import time

def harvest_rpde_feed_resilient(feed_url, provider_name, feed_type, max_retries=5):
    """
    Same as harvest_rpde_feed, but for feeds that 403 aggressively.
    On a 403, waits longer each retry (10s, 20s, 30s...) instead of
    giving up after one attempt.
    """
    session = requests.Session()
    session.headers.update(HEADERS)

    all_items = []
    current_url = feed_url
    page = 0
    seen_urls = set()
    print(f"\n  Harvesting {provider_name} / {feed_type}\n  Start: {feed_url}")

    while True:
        page += 1
        retries = 0
        while retries <= max_retries:
            try:
                resp = session.get(current_url, timeout=REQUEST_TIMEOUT)
                if resp.status_code == 403:
                    retries += 1
                    wait = 10 * retries
                    print(f"    [403] page {page}, retry {retries}/{max_retries} -- waiting {wait}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as e:
                print(f"    [ERROR] page {page}: {e} -- stopping.")
                return all_items
        else:
            print(f"    Gave up after {max_retries} retries on page {page}. Stopping.")
            return all_items

        items = data.get("items", [])
        next_url = data.get("next")
        kept = [it for it in items if it.get("state") != "deleted" and it.get("data")]
        all_items.extend(kept)

        if page % 20 == 0 or not items:
            print(f"    page {page}: +{len(kept)} (total: {len(all_items)})")
        if not items and next_url == current_url:
            print(f"    Reached live edge. Total: {len(all_items)}")
            break
        if not next_url or next_url in seen_urls:
            break
        seen_urls.add(current_url)
        current_url = next_url
        time.sleep(8.0)  # slower baseline pace too

    return all_items

In [62]:
for feed_type in ["SessionSeries", "Slot"]:
    feed_url = provider_feeds["Serco Leisure"][feed_type]
    items = harvest_rpde_feed_resilient(feed_url, "Serco Leisure", feed_type)
    if items:
        save_items(items, "Serco Leisure", feed_type)


  Harvesting Serco Leisure / SessionSeries
  Start: https://serco-openactive.legendonlineservices.co.uk/api/sessions
    [403] page 4, retry 1/5 -- waiting 10s...
    [403] page 6, retry 1/5 -- waiting 10s...
    [403] page 8, retry 1/5 -- waiting 10s...
    [403] page 8, retry 2/5 -- waiting 20s...
    page 20: +9 (total: 9)
    [403] page 26, retry 1/5 -- waiting 10s...
    [403] page 38, retry 1/5 -- waiting 10s...
    [403] page 40, retry 1/5 -- waiting 10s...
    page 40: +100 (total: 2009)
    [403] page 46, retry 1/5 -- waiting 10s...
    [403] page 50, retry 1/5 -- waiting 10s...
    [403] page 56, retry 1/5 -- waiting 10s...
    [403] page 56, retry 2/5 -- waiting 20s...
    [403] page 58, retry 1/5 -- waiting 10s...
    page 60: +100 (total: 4009)
    [403] page 62, retry 1/5 -- waiting 10s...
    [403] page 66, retry 1/5 -- waiting 10s...
    [403] page 70, retry 1/5 -- waiting 10s...
    [403] page 72, retry 1/5 -- waiting 10s...
    [403] page 74, retry 1/5 -- waiting 10s

KeyboardInterrupt: 